# Projeto Final Mina — Fase 03: Análise Crítica, Correção de Leakage e Modelo Definitivo

## Objetivo
Este notebook tem dois papéis:
1. **Documentar e analisar os resultados do Notebook 02** com olhar crítico
2. **Corrigir o Data Leakage** identificado na feature `dontgo_8H` e reprocessar o modelo com features honestas


## 1. Revisão Crítica dos Resultados do Notebook 02

### 1.1 O Problema: Data Leakage Catastrófico em `dontgo_8H`

> [!CAUTION]
> **O Notebook 02 sofreu de Data Leakage.** A feature `dontgo_8H` é a soma de eventos `Is_Dont_Go` na **mesma janela de 8H** usada como target. Ou seja, o modelo foi treinado com o gabarito da prova na mão.

**Por que isso acontece:**
- `Is_Dont_Go` (target) = "houve pelo menos 1 Don't Go nesta janela de 8H?"
- `dontgo_8H` (feature) = "quantos Don't Gos ocorreram nesta janela de 8H?"

São a mesma informação, derivadas da mesma janela temporal. O modelo aprendeu a "prever" o presente com o presente — não o futuro com o passado.

**Por que resultados altos no NB02 são suspeitos:**
Um modelo com leakage consegue F1 e AUC altíssimos na fase de teste *porque os dados de teste também têm leakage*. No mundo real, quando o alarme ainda não ocorreu (é exatamente o que queremos prever), `dontgo_8H = 0` para todos — e o modelo colapsaria.

### 1.2 O Modelo Correto: Previsão com Janelas Defasadas (Shift +1)

A pergunta correta de negócio é:
> *"Com base no comportamento das últimas horas, essa máquina vai falhar no próximo turno?"*

Para isso, precisamos que todas as features sejam calculadas com `.shift(1)` — ou seja, usam apenas informação do passado para prever o futuro imediato.


In [ ]:
# Simulação dos resultados do NB02 para registro (preencha com os valores reais obtidos)
import pandas as pd

resultados_nb02 = pd.DataFrame([
    {'Modelo': 'DummyClassifier (Baseline)', 'Frota': 'Combinado', 'F1': None, 'AUC': None, 'Obs': 'Referência ingênua'},
    {'Modelo': 'IsolationForest',            'Frota': 'Combinado', 'F1': None, 'AUC': None, 'Obs': 'Sem leakage — não usa dontgo_8H'},
    {'Modelo': 'RandomForest',               'Frota': 'Combinado', 'F1': None, 'AUC': None, 'Obs': 'COM leakage'},
    {'Modelo': 'XGBoost',                    'Frota': 'Combinado', 'F1': None, 'AUC': None, 'Obs': 'COM leakage'},
    {'Modelo': 'LightGBM',                   'Frota': 'Combinado', 'F1': None, 'AUC': None, 'Obs': 'COM leakage'},
    {'Modelo': 'CatBoost',                   'Frota': 'Combinado', 'F1': None, 'AUC': None, 'Obs': 'COM leakage'},
    {'Modelo': 'XGBoost_CA',                 'Frota': 'CA',        'F1': None, 'AUC': None, 'Obs': 'COM leakage'},
    {'Modelo': 'XGBoost_PE',                 'Frota': 'PE',        'F1': None, 'AUC': None, 'Obs': 'COM leakage'},
])

# ⚠️ Preencha os valores de F1 e AUC com os resultados reais do NB02
# resultados_nb02.loc[resultados_nb02['Modelo'] == 'XGBoost', 'F1'] = 0.XXXX

print("=== RESULTADOS DO NOTEBOOK 02 (Para Referência Histórica) ===")
print("ATENÇÃO: Modelos marcados 'COM leakage' possuem dontgo_8H como feature.")
print("Os valores de F1/AUC são otimistas e não refletem performance real.")
display(resultados_nb02)


## 2. Ingestão e Feature Engineering Corrigida

Reproduzimos todo o pipeline do NB01/NB02, mas **com correção do leakage** via `.shift(1)`:
todas as features são calculadas na janela de 8H *anterior* ao período a ser previsto.

Adicionamos também:
- **Target Encoding do Operador**: taxa histórica de Don't Go calculada **apenas no bloco de treino**, depois aplicada a validação e teste (evita leakage do operador)
- **Feature de Prefixo**: CA=0, PE=1 para o experimento combinado


In [ ]:
import pandas as pd
import numpy as np
import glob
import re
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# ── Ingestão ──
cols = ['TAG', 'Data_Evento', 'Is_Dont_Go', 'Alarme', 'Tipo', 'Nome_Operador_Anon', 'Criticidade']
dfs = [pd.read_parquet(f, columns=cols)
       for f in glob.glob(r'c:\\TT\\AntiGravity\\Vale_\\data\\raw\\Base\\datasets\\telemetria\\*.parquet')]
df = pd.concat(dfs, ignore_index=True)

# ── Tratamentos do EDA ──
df['TAG_Limpa'] = df['TAG'].replace({'CA5926': 'CA65926', 'CA5927': 'CA65927'})
df['Prefixo']   = df['TAG_Limpa'].str[:2]
df['Data_Evento'] = pd.to_datetime(df['Data_Evento'])
df['Mes'] = df['Data_Evento'].dt.month
df['Dia']  = df['Data_Evento'].dt.day

# Expurgo Cirúrgico
df = df[~((df['TAG_Limpa'] == 'PE3798') & (df['Mes'] == 6) & (df['Dia'] == 29))]
df = df[~((df['TAG_Limpa'] == 'PE3797') & (df['Mes'] == 1) & (df['Dia'] == 12))]
df = df[~((df['TAG_Limpa'] == 'CA65932') & (df['Mes'] == 3) & (df['Dia'] == 26))]

df = df.sort_values(['TAG_Limpa', 'Data_Evento']).reset_index(drop=True)
print(f"Dataset limpo: {len(df):,} registros | {df['TAG_Limpa'].nunique()} equipamentos")
print(f"Proporção de Don't Go: {df['Is_Dont_Go'].mean()*100:.4f}%")


## 3. Feature Engineering Corrigida (Sem Leakage)

**Regra de ouro**: nenhuma feature pode conter informação da janela atual (t).
Todas as contagens e taxas são calculadas nas janelas passadas e depois deslocadas com `.shift(1)`.

Isso garante que para prever o período `[t, t+8H]`, o modelo só vê informações até `[t-8H]`.


In [ ]:
top_alarmes = df['Alarme'].value_counts().head(50).index.tolist()

print("Gerando features com shift(1) — sem leakage...")
blocos = []

for tag, grupo in df.groupby('TAG_Limpa'):
    g = grupo.set_index('Data_Evento').sort_index()

    base = pd.DataFrame({
        'count_8H':   g['Is_Dont_Go'].resample('8h').count(),
        'dontgo_8H':  g['Is_Dont_Go'].resample('8h').sum(),
        'critico_8H': g['Criticidade'].resample('8h').apply(
                          lambda x: (x.str.upper() == 'CRITICO').sum()),
        'Is_Dont_Go': g['Is_Dont_Go'].resample('8h').sum(),  # target
    }).fillna(0)

    # Rolling calendarizado (IGUAL ao NB02 — legítimo)
    base['count_24H']   = base['count_8H'].rolling(3, min_periods=1).sum()
    base['dontgo_24H']  = base['dontgo_8H'].rolling(3, min_periods=1).sum()
    base['taxa_24H']    = base['dontgo_24H'] / base['count_24H'].clip(lower=1)
    base['critico_24H'] = base['critico_8H'].rolling(3, min_periods=1).sum()

    base['count_72H']   = base['count_8H'].rolling(9, min_periods=1).sum()
    base['dontgo_72H']  = base['dontgo_8H'].rolling(9, min_periods=1).sum()
    base['taxa_72H']    = base['dontgo_72H'] / base['count_72H'].clip(lower=1)
    base['critico_72H'] = base['critico_8H'].rolling(9, min_periods=1).sum()

    # ── CORREÇÃO DO LEAKAGE: shift(1) em TODAS as features ──
    # Deslocamos as features 1 janela para o futuro → o modelo prevê [t] usando [t-1, t-2, ...]
    feature_cols_shift = [c for c in base.columns if c != 'Is_Dont_Go']
    base[feature_cols_shift] = base[feature_cols_shift].shift(1)

    base['TAG_Limpa'] = tag
    base['Prefixo']   = tag[:2]
    blocos.append(base)

features = pd.concat(blocos).reset_index().rename(columns={'index': 'Data_Evento'})
# Remover primeira linha de cada equipamento (shift gera NaN na primeira janela)
features = features.dropna(subset=['count_8H']).copy()
features['Is_Dont_Go_bin'] = (features['Is_Dont_Go'] > 0).astype(int)

print(f"Features (corrigidas): {features.shape[0]:,} linhas x {features.shape[1]} colunas")
print(f"Proporção de Don't Go no target (janelas): {features['Is_Dont_Go_bin'].mean()*100:.2f}%")
display(features.head(3))


## 4. Target Encoding do Operador (Sem Leakage)

A taxa histórica de Don't Go por operador é calculada **exclusivamente sobre o bloco de treino** e depois mapeada para validação e teste.
Usar dados do futuro (validação/teste) para calcular a taxa seria mais um tipo de leakage.


In [ ]:
# Merge do operador por TAG+janela (modo do operador mais frequente na janela)
op_por_janela = df.groupby(['TAG_Limpa', pd.Grouper(key='Data_Evento', freq='8h')])[
    'Nome_Operador_Anon'].agg(lambda x: x.value_counts().index[0] if len(x) > 0 else 'DESCONHECIDO').reset_index()
op_por_janela.rename(columns={'Data_Evento': 'Data_Evento', 'Nome_Operador_Anon': 'Operador'}, inplace=True)

features = features.merge(op_por_janela, on=['TAG_Limpa', 'Data_Evento'], how='left')
features['Operador'] = features['Operador'].fillna('DESCONHECIDO')

print(f"Operadores presentes nas janelas: {features['Operador'].nunique()}")
display(features[['TAG_Limpa', 'Data_Evento', 'Operador', 'Is_Dont_Go_bin']].head(3))


## 5. Split Temporal e Aplicação do Target Encoding

Split cronológico idêntico ao NB02:
- Treino: meses 1–4 | Validação: mês 5 | Teste: mês 6


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

COLS_EXCLUIR = ['TAG_Limpa', 'Data_Evento', 'Is_Dont_Go', 'Is_Dont_Go_bin',
                'Operador', 'TAG', 'Tipo', 'Alarme', 'Criticidade', 'Prefixo', 'Mes', 'Dia']

def preparar_split_v2(df_f, label=''):
    df_f = df_f.sort_values('Data_Evento').copy()

    treino    = df_f[df_f['Data_Evento'].dt.month <= 4]
    validacao = df_f[df_f['Data_Evento'].dt.month == 5]
    teste     = df_f[df_f['Data_Evento'].dt.month == 6]

    # Target Encoding do Operador (calculado APENAS no treino)
    enc_op = treino.groupby('Operador')['Is_Dont_Go_bin'].mean().to_dict()
    global_mean = treino['Is_Dont_Go_bin'].mean()
    for split in [treino, validacao, teste]:
        split['op_taxa_dontgo'] = split['Operador'].map(enc_op).fillna(global_mean)

    # Prefixo como feature numérica
    for split in [treino, validacao, teste]:
        split['is_pe'] = (split['TAG_Limpa'].str.startswith('PE')).astype(int)

    feat_cols = [c for c in treino.columns if c not in COLS_EXCLUIR]

    X_tr  = treino[feat_cols].fillna(0)
    y_tr  = treino['Is_Dont_Go_bin']
    X_val = validacao[feat_cols].fillna(0)
    y_val = validacao['Is_Dont_Go_bin']
    X_te  = teste[feat_cols].fillna(0)
    y_te  = teste['Is_Dont_Go_bin']

    print(f"{label} Features: {feat_cols}")
    print(f"{label} Treino:    {X_tr.shape} | Don'tGo: {y_tr.sum()} ({y_tr.mean()*100:.2f}%)")
    print(f"{label} Validação: {X_val.shape} | Don'tGo: {y_val.sum()} ({y_val.mean()*100:.2f}%)")
    print(f"{label} Teste:     {X_te.shape} | Don'tGo: {y_te.sum()} ({y_te.mean()*100:.2f}%)")
    return X_tr, y_tr, X_val, y_val, X_te, y_te, feat_cols

print("=== FROTA COMBINADA ===")
X_tr, y_tr, X_val, y_val, X_te, y_te, feat_cols = preparar_split_v2(features, '[COMB]')

scale_pos = int((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
print(f"\nscale_pos_weight: {scale_pos}")
tscv = TimeSeriesSplit(n_splits=5)


## 6. Modelos Supervisionados (Features Honestas)

Com o leakage corrigido, os resultados de F1 e AUC esperados serão **menores** que os do NB02.
Isso é normal e esperado — estamos agora medindo performance real.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import f1_score, roc_auc_score, classification_report, ConfusionMatrixDisplay
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna, joblib, os
optuna.logging.set_verbosity(optuna.logging.WARNING)
import numpy as np

os.makedirs(r'c:\\TT\\AntiGravity\\Vale_\\models', exist_ok=True)
resultados = []

def avaliar(nome, modelo, X_t, y_t, frota='Combinado'):
    y_p = modelo.predict(X_t)
    y_prob = modelo.predict_proba(X_t)[:, 1] if hasattr(modelo, 'predict_proba') else y_p.astype(float)
    f1  = f1_score(y_t, y_p, zero_division=0)
    auc = roc_auc_score(y_t, y_prob) if y_t.nunique() > 1 else 0.0
    print(f"[{nome}] F1={f1:.4f} | AUC={auc:.4f}")
    print(classification_report(y_t, y_p, target_names=['Normal', "Don't Go"], zero_division=0))
    resultados.append({'Modelo': nome, 'Frota': frota, 'F1': f1, 'AUC': auc})

# ── Baseline ──
print("=== Baseline ===")
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_tr, y_tr)
avaliar('Baseline', dummy, X_te, y_te)

# ── XGBoost (RandomizedSearch) ──
print("\n=== XGBoost ===")
xgb_params = {
    'n_estimators': [300, 500, 1000],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'scale_pos_weight': [scale_pos, scale_pos // 2],
}
xgb = RandomizedSearchCV(
    XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss', use_label_encoder=False),
    xgb_params, n_iter=30, cv=tscv, scoring='f1', random_state=42, n_jobs=-1
)
xgb.fit(X_tr, y_tr)
print(f"Best params: {xgb.best_params_}")
avaliar('XGBoost', xgb.best_estimator_, X_te, y_te)

# ── LightGBM ──
print("\n=== LightGBM ===")
lgb_params = {
    'n_estimators': [300, 500, 1000],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 63, 127],
    'class_weight': ['balanced'],
}
lgb = RandomizedSearchCV(
    LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    lgb_params, n_iter=30, cv=tscv, scoring='f1', random_state=42, n_jobs=-1
)
lgb.fit(X_tr, y_tr)
print(f"Best params: {lgb.best_params_}")
avaliar('LightGBM', lgb.best_estimator_, X_te, y_te)

# ── CatBoost com Optuna ──
print("\n=== CatBoost + Optuna ===")
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, scale_pos),
        'random_seed': 42, 'verbose': 0,
    }
    scores = []
    for tr_idx, val_idx in tscv.split(X_tr):
        m = CatBoostClassifier(**params)
        m.fit(X_tr.iloc[tr_idx], y_tr.iloc[tr_idx], verbose=0)
        scores.append(f1_score(y_tr.iloc[val_idx], m.predict(X_tr.iloc[val_idx]), zero_division=0))
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective_cat, n_trials=50, show_progress_bar=True)
best_cat = CatBoostClassifier(**study.best_params, random_seed=42, verbose=0)
best_cat.fit(X_tr, y_tr)
avaliar('CatBoost', best_cat, X_te, y_te)


## 7. Experimento Separado CA vs PE


In [ ]:
for prefixo in ['CA', 'PE']:
    print(f"\n{'='*50}")
    print(f"=== {prefixo} ===")
    df_frota = features[features['TAG_Limpa'].str.startswith(prefixo)].copy()
    Xtr_f, ytr_f, _, _, Xte_f, yte_f, _ = preparar_split_v2(df_frota, f'[{prefixo}]')
    sp = int((ytr_f == 0).sum() / max((ytr_f == 1).sum(), 1))
    xgb_f = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss',
                           use_label_encoder=False, scale_pos_weight=sp,
                           n_estimators=500, max_depth=6, learning_rate=0.05)
    xgb_f.fit(Xtr_f, ytr_f)
    avaliar(f'XGBoost_{prefixo}', xgb_f, Xte_f, yte_f, frota=prefixo)


## 8. Tabela Comparativa Final (NB02 vs NB03)


In [ ]:
from IPython import display
df_res = pd.DataFrame(resultados).sort_values('F1', ascending=False).reset_index(drop=True)

print("=== RESULTADOS NB03 (Features Honestas — Sem Leakage) ===")
display(df_res.style.background_gradient(cmap='RdYlGn', subset=['F1', 'AUC']))

melhor = df_res.iloc[0]
print(f"\nMelhor modelo (NB03): {melhor['Modelo']} | F1={melhor['F1']:.4f} | AUC={melhor['AUC']:.4f}")
print("\n=== ANÁLISE COMPARATIVA ===")
print("NB02 tinha Data Leakage (dontgo_8H = target). Resultados eram inflados.")
print("NB03 usa shift(1) + target encoding de operador calculado apenas no treino.")
print("A diferença de F1 entre NB02 e NB03 é a medida do quanto o leakage inflava os resultados.")


## 9. Interpretabilidade — SHAP Values (Modelo Correto)

Com o modelo honesto, o SHAP agora revela quais features do **passado** mais influenciam a previsão de falha no **próximo turno**.


In [ ]:
import shap

explainer = shap.TreeExplainer(best_cat)
shap_values = explainer.shap_values(X_te)

print("=== SHAP — Importância Global ===")
shap.summary_plot(shap_values, X_te, plot_type='bar', max_display=15, show=True)

print("\n=== SHAP — Beeswarm (Impacto e Direção) ===")
shap.summary_plot(shap_values, X_te, max_display=15, show=True)


## 10. Persistência do Modelo Definitivo


In [ ]:
import joblib

caminho = r'c:\\TT\\AntiGravity\\Vale_\\models\\best_model_v2_no_leakage.pkl'
joblib.dump(best_cat, caminho)
print(f"Modelo salvo: {caminho}")
print(f"Features: {feat_cols}")
print("\nPróximo passo: Relatório Final / Apresentação")
